# Monte Carlo Simulation: OLS vs Newey–West HAC under AR(4) Autocorrelation

Goal: Evaluate the performance of ordinary least squares (OLS) standard errors
compared to heteroskedasticity-and-autocorrelation consistent (HAC)
Newey–West estimators when residuals follow an AR(4) process.


In [321]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from scipy.stats import norm, chi2, t


In [322]:
# Simulation parameters
np.random.seed(2025)
R = 1000                # number of replications
T = 100                 # sample size
betas_true = np.array([0.0, 1.0, 0.5, -0.5])
phi = np.array([0.4, -0.2, 0.15, -0.05])  # AR(4) coefficients
bandwidths = [0, 1, 4, int(4 * (T/100)**(2/9))]  # HAC lag lengths


In [323]:
def simulate_ar4(T, phi, sigma=1.0):
    epsilon = np.random.normal(0, sigma, T)
    u = np.zeros(T)
    for t in range(4, T):
        u[t] = phi[0]*u[t-1] + phi[1]*u[t-2] + phi[2]*u[t-3] + phi[3]*u[t-4] + epsilon[t]
    return u

def one_replication(T, phi, betas, bandwidths):
    """
    Run one Monte Carlo replication comparing OLS and Newey-West estimators.

    Returns a dictionary with:
    betahat, sandwich variance, beta variance, confidence intervals, and p-values.
    """

    # --- 1️⃣ Simulate regressors ---
    X = np.column_stack([
        np.ones(T),
        np.random.normal(size=T),
        np.random.normal(size=T),
        np.random.normal(size=T)
    ])

    # --- 2️⃣ Simulate AR(4) errors ---
    u = simulate_ar4(T,phi)

    # --- 3️⃣ Generate dependent variable ---
    y = np.matmul(X,betas) + u

    # --- 4️⃣ Fit OLS model ---
    model = sm.OLS(y, X).fit()
    betahat = np.linalg.inv(X.T @ X) @ X.T @ y
    k = len(betahat)
    df = T - k

    # --- 5️⃣ Initialize results dictionary ---
    results = {}

    # --- 6️⃣ Compute OLS quantities ---

    residuals = (y - X @ betahat)
    SSR = np.dot(residuals,residuals)
    SST = np.dot(y - np.mean(y), y -np.mean(y))
    R2_ols = 1 - SSR / SST
    R2_adj_ols = 1 - (SSR / df) / (SST / (T-1))
    s2_ols = SSR / df
    cov_beta_ols = s2_ols * np.linalg.inv(X.T @ X)
    se_beta_ols = np.sqrt(np.diag(cov_beta_ols))
    t_stats_ols = betahat / se_beta_ols
    p_value_ols = 2 * (1 - t.cdf(np.abs(t_stats_ols),df))
    z_95 = norm.ppf(1-0.05/2)
    ci_asymp_ols_95 = np.column_stack([betahat - z_95*se_beta_ols, betahat + z_95*se_beta_ols])


    results['OLS'] = {
        'betahat': betahat,
        's2': s2_ols,
        'R2': R2_ols,
        'R2_adj': R2_adj_ols,
        'cov_beta': cov_beta_ols,
        'se_beta': se_beta_ols,
        't_stats': t_stats_ols,
        'p_value': p_value_ols,
        'conf_int_95': ci_asymp_ols_95,
    }

    # --- 7️⃣ Compute Newey–West quantities ---
    for m in bandwidths:
        cov_nw = cov_hac(model, nlags=m)
        se_nw = np.sqrt(np.diag(cov_nw))
        t_nw = betahat / se_nw
        p_nw = 2 * (1 - norm.cdf(np.abs(t_nw)))
        ci_nw_95 = np.column_stack([betahat - z_95*se_nw, betahat + z_95*se_nw])

        results[f'NW({m})'] = {
            'betahat': betahat,
            'var_sandwich': cov_nw,
            'cov_beta': np.diag(cov_nw),
            'conf_int_95': ci_nw_95,
            'p_value': p_nw
        }

    return results



In [324]:
results = one_replication(T, phi, betas_true, bandwidths)

In [325]:
np.set_printoptions(formatter={'float_kind': '{:0.3}'.format})
results["OLS"]["conf_int_95"]

array([[-0.46, -0.0713],
       [0.74, 1.16],
       [0.171, 0.57],
       [-0.77, -0.369]])

In [326]:
for m in bandwidths:
  print(f"NW({m})\n", results[f"NW({m})"]["conf_int_95"])

NW(0)
 [[-0.468 -0.0636]
 [0.76 1.14]
 [0.156 0.585]
 [-0.763 -0.376]]
NW(1)
 [[-0.493 -0.0382]
 [0.777 1.12]
 [0.155 0.586]
 [-0.756 -0.382]]
NW(4)
 [[-0.515 -0.0168]
 [0.791 1.11]
 [0.141 0.6]
 [-0.744 -0.395]]
NW(4)
 [[-0.515 -0.0168]
 [0.791 1.11]
 [0.141 0.6]
 [-0.744 -0.395]]


In [327]:
np.set_printoptions(formatter={'float_kind': '{:0.3e}'.format})
results["OLS"]["p_value"]

array([8.695e-03, 3.464e-14, 4.520e-04, 2.372e-07])

In [328]:
for m in bandwidths:
  print(results[f"NW({m})"]["p_value"])

[9.980e-03 0.000e+00 7.046e-04 7.817e-09]
[2.207e-02 0.000e+00 7.646e-04 2.398e-09]
[3.639e-02 0.000e+00 1.523e-03 1.710e-10]
[3.639e-02 0.000e+00 1.523e-03 1.710e-10]


In [337]:
def montecarlo_analysis(T, phi, betas, bandwidths, n_rep=10000):
    """Run Monte Carlo to check CI coverage and p-values"""
    K = len(betas)
    coverage_ols = np.zeros(K)
    coverage_nw = {m: np.zeros(K) for m in bandwidths}

    for _ in range(n_rep):
        res = one_replication(T, phi, betas, bandwidths)

        # OLS coverage
        ci = res['OLS']['conf_int_95']
        coverage_ols += ((ci[:,0] <= betas) & (betas <= ci[:,1])).astype(int)

        # NW coverage
        for m in bandwidths:
            ci = res[f'NW({m})']['conf_int_95']
            coverage_nw[m] += ((ci[:,0] <= betas) & (betas <= ci[:,1])).astype(int)

    # Convert to probabilities
    coverage_ols /= n_rep
    coverage_nw = {m: cov/n_rep for m, cov in coverage_nw.items()}

    return coverage_ols, coverage_nw

# Example usage
T = 200
phi = [0.3, 0.2, 0.1, 0.05]
betas = np.array([1.0, 0.5, -0.3, 0.2])
bandwidths = [0, 1, int(4 * (T/100)**(2/9))]

coverage_ols, coverage_nw = montecarlo_analysis(T, phi, betas, bandwidths)

np.set_printoptions(formatter={'float_kind': '{:0.3}'.format})
print("OLS coverage:", coverage_ols)
print("Newey-West coverage:", coverage_nw)

OLS coverage: [0.574 0.946 0.949 0.947]
Newey-West coverage: {0: array([0.574, 0.944, 0.942, 0.943]), 1: array([0.655, 0.942, 0.942, 0.943]), 4: array([0.781, 0.936, 0.938, 0.937])}


In [335]:
print(int(4 * (T/100)**(2/9)))

4
